У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [3]:
df = pd.read_csv('customer_segmentation_train.csv')

df

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A
...,...,...,...,...,...,...,...,...,...,...,...
8063,464018,Male,No,22,No,NaN,0.0,Low,7.0,Cat_1,D
8064,464685,Male,No,35,No,Executive,3.0,Low,4.0,Cat_4,D
8065,465406,Female,No,33,Yes,Healthcare,1.0,Low,1.0,Cat_6,D
8066,467299,Female,No,27,Yes,Healthcare,1.0,Low,4.0,Cat_6,B


In [4]:
df.describe()

,ID,Age,Work_Experience,Family_Size
count,8068.000000,8068.000000,7239.000000,7733.000000
mean,463479.214551,43.466906,2.641663,2.850123
std,2595.381232,16.711696,3.406763,1.531413
min,458982.000000,18.000000,0.000000,1.000000
25%,461240.750000,30.000000,0.000000,2.000000
50%,463472.500000,40.000000,1.000000,3.000000
75%,465744.250000,53.000000,4.000000,4.000000
max,467974.000000,89.000000,14.000000,9.000000


In [5]:
df = df.drop('ID', axis=1)

In [6]:
df.isna().sum()

Gender               0
Ever_Married       140
Age                  0
Graduated           78
Profession         124
Work_Experience    829
Spending_Score       0
Family_Size        335
Var_1               76
Segmentation         0
dtype: int64

In [7]:
X = df.drop('Segmentation', axis=1)
y = df['Segmentation']

In [8]:
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes('object').columns.tolist()
numeric_cols, categorical_cols

(['Age', 'Work_Experience', 'Family_Size'],
 ['Gender',
  'Ever_Married',
  'Graduated',
  'Profession',
  'Spending_Score',
  'Var_1'])

In [9]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()) 
])

In [13]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [15]:
X_ = preprocessor.fit_transform(X)

In [16]:
col_names = preprocessor.get_feature_names_out()

In [17]:
X_transformed = pd.DataFrame(X_, columns=col_names, index=X.index)

In [18]:
X_transformed

,num__Age,num__Work_Experience,num__Family_Size,cat__Gender_Female,cat__Gender_Male,cat__Ever_Married_No,cat__Ever_Married_Yes,cat__Graduated_No,cat__Graduated_Yes,cat__Profession_Artist,...,cat__Spending_Score_Average,cat__Spending_Score_High,cat__Spending_Score_Low,cat__Var_1_Cat_1,cat__Var_1_Cat_2,cat__Var_1_Cat_3,cat__Var_1_Cat_4,cat__Var_1_Cat_5,cat__Var_1_Cat_6,cat__Var_1_Cat_7
0,-1.284623,-0.508763,0.767001,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,-0.327151,0.000000,0.099972,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,1.408268,-0.508763,-1.234085,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.408268,-0.818671,-0.567056,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.207467,0.000000,2.101059,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8063,-1.284623,-0.818671,2.768088,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
8064,-0.506677,0.111051,0.767001,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
8065,-0.626361,-0.508763,-1.234085,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8066,-0.985413,-0.508763,0.767001,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, y, test_size=0.2, random_state=42, stratify=y
)

In [72]:
y.value_counts()

D    2268
A    1972
C    1970
B    1858
Name: Segmentation, dtype: int64

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [59]:
all_cols = X_train.columns.tolist()
all_cols

['num__Age',
 'num__Work_Experience',
 'num__Family_Size',
 'cat__Gender_Female',
 'cat__Gender_Male',
 'cat__Ever_Married_No',
 'cat__Ever_Married_Yes',
 'cat__Graduated_No',
 'cat__Graduated_Yes',
 'cat__Profession_Artist',
 'cat__Profession_Doctor',
 'cat__Profession_Engineer',
 'cat__Profession_Entertainment',
 'cat__Profession_Executive',
 'cat__Profession_Healthcare',
 'cat__Profession_Homemaker',
 'cat__Profession_Lawyer',
 'cat__Profession_Marketing',
 'cat__Spending_Score_Average',
 'cat__Spending_Score_High',
 'cat__Spending_Score_Low',
 'cat__Var_1_Cat_1',
 'cat__Var_1_Cat_2',
 'cat__Var_1_Cat_3',
 'cat__Var_1_Cat_4',
 'cat__Var_1_Cat_5',
 'cat__Var_1_Cat_6',
 'cat__Var_1_Cat_7']

In [66]:
from imblearn.over_sampling import SMOTE

numeric_cols_ = [col for col in X_train.columns 
                     if not any(cat_col in col for cat_col in categorical_cols)]

X_train_num = X_train[numeric_cols_]

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_num, y_train)

print("Після SMOTE:")
print(y_smote.value_counts())
X_smote

Після SMOTE:
A    1814
B    1814
C    1814
D    1814
Name: Segmentation, dtype: int64


C:\Users\Admin\anaconda3\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


,num__Age,num__Work_Experience,num__Family_Size
0,-0.686203,1.970495,-1.234085
1,1.707478,0.000000,-0.567056
2,-0.626361,-0.508763,0.767001
3,0.271270,-0.818671,2.101059
4,-0.925571,1.970495,-1.234085
...,...,...,...
7251,1.527952,-0.818671,-0.567056
7252,-0.633699,0.552522,0.767001
7253,0.211428,-0.508763,0.767001
7254,-0.147625,-0.508763,-0.567056


In [61]:
cat_feature_indeces = [i for i, col in enumerate(all_cols) 
                       if not any(num_col in col for num_col in numeric_cols)]

In [67]:
from imblearn.over_sampling import SMOTENC

smotenc = SMOTENC(
    categorical_features=cat_feature_indeces,
    random_state=42
)

X_smotenc, y_smotenc = smotenc.fit_resample(X_train, y_train)

print("Після SMOTENC:")
print(y_res_smotenc.value_counts())

C:\Users\Admin\anaconda3\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
C:\Users\Admin\anaconda3\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Після SMOTENC:
A    1814
B    1814
C    1814
D    1814
Name: Segmentation, dtype: int64


,num__Age,num__Work_Experience,num__Family_Size,cat__Gender_Female,cat__Gender_Male,cat__Ever_Married_No,cat__Ever_Married_Yes,cat__Graduated_No,cat__Graduated_Yes,cat__Profession_Artist,...,cat__Spending_Score_Average,cat__Spending_Score_High,cat__Spending_Score_Low,cat__Var_1_Cat_1,cat__Var_1_Cat_2,cat__Var_1_Cat_3,cat__Var_1_Cat_4,cat__Var_1_Cat_5,cat__Var_1_Cat_6,cat__Var_1_Cat_7
0,-0.686203,1.970495,-1.234085,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.707478,0.000000,-0.567056,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.626361,-0.508763,0.767001,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.271270,-0.818671,2.101059,1.0,0.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.925571,1.970495,-1.234085,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7251,1.696748,-0.818671,-0.567056,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7252,-0.564824,1.265897,0.383143,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7253,0.304343,-0.311277,0.508081,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7254,-0.147625,-0.508763,-0.567056,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

я вибрала F1-score, бо він враховує всі класи однаково та показує, чи модель покращила мінорні класи

In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score

ovr_model = OneVsRestClassifier(
    LogisticRegression(max_iter=2000)
)

ovr_model.fit(X_train, y_train)
y_pred_orig = ovr_model.predict(X_test)

print(classification_report(y_test, y_pred_orig))

macro_f1_orig = f1_score(y_test, y_pred_orig, average='macro')
print( macro_f1_orig)

              precision    recall  f1-score   support

           A       0.42      0.45      0.43       394
           B       0.41      0.17      0.25       372
           C       0.49      0.62      0.55       394
           D       0.65      0.76      0.70       454

    accuracy                           0.52      1614
   macro avg       0.49      0.50      0.48      1614
weighted avg       0.50      0.52      0.49      1614

0.48237445697703196


In [70]:
ovr_smote = OneVsRestClassifier(
    LogisticRegression(max_iter=2000)
)
X_test_numeric = X_test[numeric_cols_]

ovr_smote.fit(X_smote, y_smote)
y_pred_smote = ovr_smote.predict(X_test_numeric)

print(classification_report(y_test, y_pred_smote))

macro_f1_smote = f1_score(y_test, y_pred_smote, average='macro')
print(macro_f1_smote)

              precision    recall  f1-score   support

           A       0.32      0.27      0.29       394
           B       0.28      0.10      0.15       372
           C       0.36      0.42      0.39       394
           D       0.49      0.76      0.60       454

    accuracy                           0.40      1614
   macro avg       0.36      0.39      0.36      1614
weighted avg       0.37      0.40      0.37      1614

0.35712573638981593


In [71]:
ovr_smt = OneVsRestClassifier(
    LogisticRegression(max_iter=2000)
)

ovr_smt.fit(X_smotenc, y_smotenc)
y_pred_smt = ovr_smt.predict(X_test)

print(classification_report(y_test, y_pred_smt))

macro_f1_smt = f1_score(y_test, y_pred_smt, average='macro')
print(macro_f1_smt)

              precision    recall  f1-score   support

           A       0.42      0.47      0.44       394
           B       0.41      0.25      0.31       372
           C       0.51      0.59      0.55       394
           D       0.67      0.72      0.70       454

    accuracy                           0.52      1614
   macro avg       0.50      0.51      0.50      1614
weighted avg       0.51      0.52      0.51      1614

0.4998353120926944


найкраща модель, яка побудована на даних після ресамплінгу SMOTE-Tomek, але між нею і моделлю на оригінальних даних суттєвої різниці немає. напевно тому, що класи не сильно перекриваються та і сам дисбаланс між класами не критичний